# Workflow Engines: Snakemake and nf-core / Nextflow

**Tier 3 – Applied Bioinformatics**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain the DAG-based execution model shared by all modern workflow engines
2. Write **production-quality Snakefiles** with config files, wildcards, params, logs, benchmarks, conda environments, and containers
3. Run, profile, and scale Snakemake on a **cluster / cloud** (SLURM, AWS, Google Cloud)
4. Understand the **Nextflow DSL2** dataflow model and write basic processes
5. Use **nf-core** community pipelines, explore their parameter space, and customise them with profiles and config files
6. Choose between Snakemake and Nextflow for a given project

---

## Contents

1. [Why Workflow Engines?](#1.-Why-Workflow-Engines?)
2. [Snakemake Fundamentals](#2.-Snakemake-Fundamentals)
3. [Snakemake — Advanced Features](#3.-Snakemake-Advanced-Features)
4. [Snakemake — Cluster and Cloud](#4.-Snakemake-Cluster-and-Cloud)
5. [Nextflow / DSL2 Fundamentals](#5.-Nextflow-/-DSL2-Fundamentals)
6. [nf-core Community Pipelines](#6.-nf-core-Community-Pipelines)
7. [Snakemake vs Nextflow Cheat-Sheet](#7.-Snakemake-vs-Nextflow)
8. [Exercises](#8.-Exercises)
9. [Summary](#9.-Summary)


<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Single-Cell Analysis](./01_single_cell_scanpy.ipynb) | [Next: Testing and CI/CD →](./03_testing_cicd.ipynb)


---

## 1. Why Workflow Engines?

### 1.1 The shell-script problem

A typical genomics project evolves into a tangle of shell scripts:

```bash
#!/bin/bash
fastqc raw/*.fastq.gz -o qc/
trimmomatic PE raw/s1_R1.fastq.gz raw/s1_R2.fastq.gz     trimmed/s1_R1.fq.gz trimmed/s1_R1_unpaired.fq.gz     trimmed/s1_R2.fq.gz trimmed/s1_R2_unpaired.fq.gz     ILLUMINACLIP:adapters.fa:2:30:10 MINLEN:36
bwa mem -t 8 ref/genome.fa trimmed/s1_R1.fq.gz trimmed/s1_R2.fq.gz |     samtools sort -o aligned/s1.bam
...
```

Problems:
- **No resume**: reruns everything even if step 2 failed halfway through step 5
- **No parallelism**: samples processed sequentially
- **No resource control**: cannot request 8 CPUs for BWA and 1 for sorting
- **Not reproducible**: hard-coded paths, tool versions not pinned
- **No provenance**: no record of which command produced which file

### 1.2 What workflow engines add

| Feature | Shell script | Snakemake | Nextflow |
|---------|-------------|-----------|----------|
| DAG-based dependency tracking | ✗ | ✓ | ✓ |
| Automatic resume / re-run | ✗ | ✓ | ✓ |
| Parallelism | manual | automatic | automatic |
| Resource management | ✗ | ✓ | ✓ |
| Conda / containers | manual | built-in | built-in |
| Cluster / cloud | manual | ✓ | ✓ |
| Provenance / audit trail | ✗ | partial | ✓ (full) |
| Community pipelines | ✗ | catalog | nf-core |


In [ ]:
# ── Illustrate a minimal pipeline DAG (text representation) ──────────────
pipeline_dag = {
    "raw FASTQ": ["FastQC", "Trimmomatic"],
    "FastQC":    ["MultiQC"],
    "Trimmomatic": ["BWA MEM"],
    "BWA MEM":   ["samtools sort"],
    "samtools sort": ["samtools index", "GATK HaplotypeCaller"],
    "samtools index": [],
    "GATK HaplotypeCaller": ["VQSR", "VEP annotation"],
    "VQSR": ["VEP annotation"],
    "VEP annotation": ["final VCF"],
    "MultiQC": ["final QC report"],
}

def topo_sort(graph):
    """Kahn's topological sort for a DAG dict."""
    in_degree = {n: 0 for n in graph}
    for node, neighbors in graph.items():
        for nb in neighbors:
            if nb not in in_degree:
                in_degree[nb] = 0
            in_degree[nb] += 1
    queue = [n for n, d in in_degree.items() if d == 0]
    order = []
    while queue:
        n = queue.pop(0)
        order.append(n)
        for nb in graph.get(n, []):
            in_degree[nb] -= 1
            if in_degree[nb] == 0:
                queue.append(nb)
    return order

order = topo_sort(pipeline_dag)
print("Execution order (topological sort):")
for i, step in enumerate(order, 1):
    print(f"  {i:>2}. {step}")


---

## 2. Snakemake Fundamentals

### 2.1 Installation

```bash
# Recommended: conda / mamba
conda create -n snake -c conda-forge -c bioconda snakemake
conda activate snake

# pip (no conda dependencies)
pip install snakemake
```

### 2.2 Core concepts

| Concept | Description |
|---------|-------------|
| **Rule** | Unit of work: defines input, output, and the shell/script command |
| **Wildcard** | `{sample}` – Snakemake infers which rule to run by matching filenames |
| **DAG** | Directed Acyclic Graph built by backward-chaining from the target |
| **Snakefile** | The recipe file (Python + rule syntax) |
| **`all` rule** | The default target rule — lists all final outputs |

### 2.3 Minimal Snakefile anatomy

```python
# Snakefile

SAMPLES = ["sample1", "sample2", "sample3"]

rule all:
    input:
        expand("results/vcf/{sample}.vcf.gz", sample=SAMPLES),
        "results/qc/multiqc_report.html"

rule fastqc:
    input:  "data/{sample}_R1.fastq.gz"
    output: "results/qc/{sample}_fastqc.html"
    shell:  "fastqc {input} --outdir results/qc/"

rule trim:
    input:
        r1 = "data/{sample}_R1.fastq.gz",
        r2 = "data/{sample}_R2.fastq.gz"
    output:
        r1 = "trimmed/{sample}_R1.fq.gz",
        r2 = "trimmed/{sample}_R2.fq.gz"
    shell:
        "fastp -i {input.r1} -I {input.r2} -o {output.r1} -O {output.r2}"

rule bwa_mem:
    input:
        r1 = "trimmed/{sample}_R1.fq.gz",
        r2 = "trimmed/{sample}_R2.fq.gz",
        ref = "ref/hg38.fa"
    output: "aligned/{sample}.bam"
    threads: 8
    shell:
        "bwa mem -t {threads} {input.ref} {input.r1} {input.r2} "
        "| samtools sort -o {output}"
```


In [ ]:
# ── Simulate Snakemake's rule-matching logic ─────────────────────────────
import re

def match_rule_output(output_pattern, target_file):
    """
    Check if a target filename matches a rule's output pattern,
    and return the wildcard dict if it does.
    Handles {wildcard} patterns like Snakemake.
    """
    # Convert Snakemake pattern to regex: {name} → (?P<name>.+)
    regex = re.sub(r'\{(\w+)\}', r'(?P<\1>[^/]+)', re.escape(output_pattern))
    regex = regex.replace('\/', '/')  # unescape slashes
    m = re.fullmatch(regex, target_file)
    if m:
        return m.groupdict()
    return None

# Define some rules (simplified)
rules = [
    ("fastqc",  "results/qc/{sample}_fastqc.html"),
    ("trim",    "trimmed/{sample}_R1.fq.gz"),
    ("bwa_mem", "aligned/{sample}.bam"),
    ("call",    "results/vcf/{sample}.vcf.gz"),
]

# Test targets
targets = [
    "results/qc/sample1_fastqc.html",
    "aligned/sample2.bam",
    "results/vcf/sample3.vcf.gz",
    "some/unknown/file.txt",
]

print(f"{'Target':<40}  {'Matching rule':<12}  Wildcards")
print('-' * 75)
for target in targets:
    matched = None
    for rule_name, pattern in rules:
        wc = match_rule_output(pattern, target)
        if wc is not None:
            matched = (rule_name, wc)
            break
    if matched:
        print(f"{target:<40}  {matched[0]:<12}  {matched[1]}")
    else:
        print(f"{target:<40}  (no match)")


---

## 3. Snakemake — Advanced Features

### 3.1 Config files

Separate parameters from logic using a YAML config file:

```yaml
# config/config.yaml
samples:
  - sample1
  - sample2
  - sample3

reference: ref/hg38.fa
adapters:  ref/adapters.fa

trimming:
  min_length: 36
  quality:    20

calling:
  min_base_quality: 20
  min_mapping_quality: 30
```

Load in Snakefile:

```python
configfile: "config/config.yaml"

SAMPLES = config["samples"]
REF     = config["reference"]
```

### 3.2 params, log, benchmark

```python
rule bwa_mem:
    input:
        r1  = "trimmed/{sample}_R1.fq.gz",
        r2  = "trimmed/{sample}_R2.fq.gz",
        ref = config["reference"]
    output:
        bam = "aligned/{sample}.bam"
    params:
        rg  = r"@RG\tID:{sample}\tSM:{sample}\tPL:ILLUMINA"
    log:
        "logs/bwa_mem/{sample}.log"
    benchmark:
        "benchmarks/bwa_mem/{sample}.tsv"
    threads: 8
    resources:
        mem_mb = 16000
    shell:
        "(bwa mem -t {threads} -R '{params.rg}' {input.ref} "
        "{input.r1} {input.r2} | samtools sort -o {output.bam}) "
        "2> {log}"
```

**log** files capture stderr/stdout for debugging without breaking Snakemake's
output-tracking.  
**benchmark** writes a TSV with wall-clock time, CPU time, and peak RSS per rule
invocation — invaluable for profiling.

### 3.3 Conda environments per rule

```python
rule fastqc:
    input:  "data/{sample}_R1.fastq.gz"
    output: "results/qc/{sample}_fastqc.html"
    conda:  "envs/qc.yaml"     # ← per-rule conda env
    shell:  "fastqc {input} --outdir results/qc/"
```

```yaml
# envs/qc.yaml
channels:
  - bioconda
  - conda-forge
  - defaults
dependencies:
  - fastqc=0.12.1
  - multiqc=1.19
```

Run with `--use-conda` to activate:
```bash
snakemake --use-conda --cores 8
```

### 3.4 Singularity / Apptainer containers

```python
rule gatk_haplotypecaller:
    input:  bam="aligned/{sample}.bam", ref=config["reference"]
    output: gvcf="gvcf/{sample}.g.vcf.gz"
    container:
        "docker://broadinstitute/gatk:4.5.0.0"
    shell:
        "gatk HaplotypeCaller -R {input.ref} -I {input.bam} "
        "-O {output.gvcf} -ERC GVCF"
```

Run with:
```bash
snakemake --use-singularity --cores 8
```

### 3.5 Checkpoints (dynamic output)

When you don't know the number of output files in advance (e.g., genome
assembly splitting), use checkpoints:

```python
checkpoint split_by_chromosome:
    input: "assembly.fa"
    output: directory("chromosomes/")
    shell: "csplit assembly.fa ... "

def aggregate_chromosomes(wildcards):
    checkpoint_output = checkpoints.split_by_chromosome.get(**wildcards).output[0]
    chroms = glob_wildcards(os.path.join(checkpoint_output, "{chrom}.fa")).chrom
    return expand("annotated/{chrom}.gff3", chrom=chroms)

rule all:
    input: aggregate_chromosomes
```


In [ ]:
# ── Build a Snakemake-style workflow graph from rules ────────────────────
from dataclasses import dataclass, field
from typing import Dict, List, Optional

@dataclass
class Rule:
    name: str
    inputs: List[str]
    outputs: List[str]
    threads: int = 1
    mem_mb: int = 1000
    has_log: bool = False
    has_benchmark: bool = False
    conda_env: Optional[str] = None
    container: Optional[str] = None

def build_dependency_graph(rules: List[Rule]) -> Dict[str, List[str]]:
    """
    Build a rule → [upstream rules] dependency graph by matching
    output patterns to input patterns (simplified exact-match version).
    """
    # Map each output pattern to its rule
    output_to_rule = {}
    for rule in rules:
        for out in rule.outputs:
            output_to_rule[out] = rule.name
    # For each rule, find which rules produce its inputs
    graph = {rule.name: [] for rule in rules}
    for rule in rules:
        for inp in rule.inputs:
            producer = output_to_rule.get(inp)
            if producer and producer != rule.name:
                graph[rule.name].append(producer)
    return graph

# ── Example variant-calling pipeline ─────────────────────────────────────
variant_pipeline = [
    Rule("fastqc",       inputs=["data/{sample}_R1.fastq.gz"],
                         outputs=["results/qc/{sample}_fastqc.html"],
                         conda_env="envs/qc.yaml"),
    Rule("trim",         inputs=["data/{sample}_R1.fastq.gz",
                                 "data/{sample}_R2.fastq.gz"],
                         outputs=["trimmed/{sample}_R1.fq.gz",
                                  "trimmed/{sample}_R2.fq.gz"],
                         conda_env="envs/fastp.yaml"),
    Rule("bwa_mem",      inputs=["trimmed/{sample}_R1.fq.gz",
                                 "trimmed/{sample}_R2.fq.gz"],
                         outputs=["aligned/{sample}.bam"],
                         threads=8, mem_mb=16000,
                         has_log=True, has_benchmark=True),
    Rule("mark_dups",    inputs=["aligned/{sample}.bam"],
                         outputs=["dedup/{sample}.bam"],
                         has_log=True, container="docker://broadinstitute/gatk:4.5.0.0"),
    Rule("haplotype",    inputs=["dedup/{sample}.bam"],
                         outputs=["gvcf/{sample}.g.vcf.gz"],
                         threads=4, has_log=True,
                         container="docker://broadinstitute/gatk:4.5.0.0"),
]

print(f"Pipeline: {len(variant_pipeline)} rules")
print()
print(f"{'Rule':<15}  {'Threads':>7}  {'RAM':>6}  {'Conda':>5}  {'Container':>9}  {'Log':>4}  {'Bench':>5}")
print('-' * 70)
for r in variant_pipeline:
    print(f"{r.name:<15}  {r.threads:>7}  {r.mem_mb:>5}M  "          f"{'✓' if r.conda_env else '':>5}  "          f"{'✓' if r.container else '':>9}  "          f"{'✓' if r.has_log else '':>4}  "          f"{'✓' if r.has_benchmark else '':>5}")


---

## 4. Snakemake — Cluster and Cloud

### 4.1 SLURM executor plugin

```bash
pip install snakemake-executor-plugin-slurm

snakemake \
  --executor slurm \
  --jobs 50 \
  --default-resources slurm_account=my_account \
  --use-conda
```

Or specify SLURM resources inside the rule:

```python
rule bwa_mem:
    ...
    resources:
        slurm_partition = "long",
        slurm_account   = "mylab",
        runtime         = 240,   # minutes
        mem_mb          = 32000,
        cpus_per_task   = 8
```

### 4.2 Google Cloud / AWS

```bash
# Google Life Sciences (deprecated in favour of Google Batch)
pip install snakemake-executor-plugin-googlebatch

snakemake \
  --executor googlebatch \
  --default-storage-provider gcs \
  --default-storage-prefix gs://my-bucket/results \
  --jobs 200

# AWS Batch
pip install snakemake-executor-plugin-aws-batch
snakemake \
  --executor aws-batch \
  --default-storage-provider s3 \
  --default-storage-prefix s3://my-bucket/results
```

### 4.3 Profile files

Profiles let you set executor-specific defaults without cluttering the Snakefile:

```yaml
# profiles/slurm/config.yaml
executor: slurm
jobs: 50
use-conda: true
default-resources:
  - slurm_account=mylab
  - mem_mb=4000
  - runtime=120
printshellcmds: true
keep-going: true
```

```bash
snakemake --profile profiles/slurm
```

### 4.4 Useful flags for production runs

| Flag | Purpose |
|------|---------|
| `--dry-run` / `-n` | Print included jobs without running |
| `--dag \| dot -Tpdf > dag.pdf` | Visualise the execution DAG |
| `--report report.html` | Generate HTML report with stats |
| `--rerun-incomplete` | Re-run any incomplete outputs |
| `--keep-going` / `-k` | Continue with independent jobs after a failure |
| `--latency-wait 60` | Wait 60s for NFS latency after a job finishes |
| `--shadow-prefix /tmp` | Isolate rule execution in a shadow directory |
| `--envvars VAR1 VAR2` | Forward env vars to cluster jobs |


In [ ]:
# ── Simulate a Snakemake dry-run output ──────────────────────────────────
import textwrap

SAMPLES = ["sample1", "sample2"]

def snakemake_dry_run(rules, samples):
    """Simulate the job list Snakemake would print for --dry-run."""
    jobs = []
    for sample in samples:
        for rule in rules:
            # Expand wildcard patterns
            inputs  = [p.replace('{sample}', sample) for p in rule.inputs]
            outputs = [p.replace('{sample}', sample) for p in rule.outputs]
            jobs.append({
                'rule': rule.name,
                'sample': sample,
                'inputs': inputs,
                'outputs': outputs,
                'threads': rule.threads,
                'mem_mb': rule.mem_mb,
            })
    return jobs

jobs = snakemake_dry_run(variant_pipeline, SAMPLES)

print(f"Job summary: {len(jobs)} jobs total")
print()
for j in jobs:
    print(f"  rule {j['rule']} (sample={j['sample']})")
    print(f"    input:   {', '.join(j['inputs'][:2])}")
    print(f"    output:  {', '.join(j['outputs'][:1])}")
    print(f"    threads: {j['threads']}  mem: {j['mem_mb']}M")
    print()

total_cpu = sum(j['threads'] for j in jobs)
total_mem = sum(j['mem_mb'] for j in jobs)
print(f"Total CPU-slots if fully parallel: {total_cpu}")
print(f"Total RAM if fully parallel: {total_mem/1000:.1f} GB")


---

## 5. Nextflow / DSL2 Fundamentals

### 5.1 Concepts

Nextflow uses a **dataflow programming model**: data moves through
**channels** and is processed by **processes**.

| Concept | Description |
|---------|-------------|
| **Process** | A computational step (like a Snakemake rule) |
| **Channel** | A queue of data items flowing between processes |
| **Workflow** | Wires processes together |
| **Module** | A reusable process definition (nf-core style) |
| **Profile** | Named configuration set (docker, singularity, cluster…) |

### 5.2 Installation

```bash
# Install Java (required)
sdk install java 17.0.9-tem   # or via package manager

# Install Nextflow
curl -s https://get.nextflow.io | bash
# or
conda install -c bioconda nextflow
```

### 5.3 DSL2 process anatomy

```nextflow
#!/usr/bin/env nextflow
nextflow.enable.dsl = 2

// Parameters (override on CLI with --param value)
params.reads   = "data/*_{R1,R2}.fastq.gz"
params.genome  = "ref/hg38.fa"
params.outdir  = "results"

// Process definition
process BWA_MEM {
    tag "$sample_id"            // label shown in logs
    label 'process_high'        // links to resource profile
    publishDir "${params.outdir}/aligned", mode: 'copy'

    conda "bioconda::bwa=0.7.17 bioconda::samtools=1.19"
    container "quay.io/biocontainers/bwa:0.7.17--h5bf99c6_8"

    input:
    tuple val(sample_id), path(reads)
    path  genome

    output:
    tuple val(sample_id), path("${sample_id}.bam"), emit: bam
    path  "versions.yml",                            emit: versions

    script:
    def (r1, r2) = reads
    """
    bwa mem \\
        -t $task.cpus \\
        -R '@RG\tID:${sample_id}\tSM:${sample_id}\tPL:ILLUMINA' \\
        $genome $r1 $r2 \\
        | samtools sort -o ${sample_id}.bam

    cat <<-END_VERSIONS > versions.yml
    "${task.process}":
        bwa: \$(bwa 2>&1 | grep -e '^Version' | sed 's/Version: //')
        samtools: \$(samtools --version | head -n1 | awk '{print \$2}')
    END_VERSIONS
    """
}

// Workflow wiring
workflow {
    reads_ch = Channel.fromFilePairs(params.reads)
    genome_ch = Channel.fromPath(params.genome)
    BWA_MEM(reads_ch, genome_ch)
    BWA_MEM.out.bam.view { sample, bam -> "Aligned: $sample → $bam" }
}
```

### 5.4 Running Nextflow

```bash
# Local execution
nextflow run main.nf -profile docker

# Resume a failed run (Nextflow caches completed tasks)
nextflow run main.nf -resume

# Specify parameters
nextflow run main.nf --reads 'data/*_R{1,2}.fastq.gz' --outdir my_results

# SLURM execution
nextflow run main.nf -profile slurm --outdir s3://bucket/results
```

### 5.5 Nextflow config file

```groovy
// nextflow.config
params {
    reads  = "data/*_{R1,R2}.fastq.gz"
    genome = "ref/hg38.fa"
    outdir = "results"
}

profiles {
    docker {
        docker.enabled = true
    }
    singularity {
        singularity.enabled    = true
        singularity.autoMounts = true
    }
    slurm {
        process.executor = 'slurm'
        process.queue    = 'long'
        process.memory   = '8 GB'
        process.cpus     = 4
        process.time     = '4h'
    }
}

process {
    withLabel: 'process_high' {
        cpus   = 16
        memory = '64 GB'
        time   = '8h'
    }
    withLabel: 'process_low' {
        cpus   = 2
        memory = '4 GB'
        time   = '1h'
    }
}
```


In [ ]:
# ── Nextflow channel operations (Python simulation) ───────────────────────
from collections import deque
from typing import Any, Callable, Iterator

class Channel:
    """
    Simplified Nextflow-like channel for demonstration.
    Real Nextflow channels use Groovy closures and reactive streams.
    """
    def __init__(self, items=None):
        self._items = list(items or [])

    @classmethod
    def fromList(cls, items):
        return cls(items)

    @classmethod
    def fromFilePairs(cls, pairs):
        """Simulate Channel.fromFilePairs: yield (id, [r1, r2]) tuples."""
        return cls(pairs)

    def map(self, func: Callable) -> 'Channel':
        return Channel(func(item) for item in self._items)

    def filter(self, func: Callable) -> 'Channel':
        return Channel(item for item in self._items if func(item))

    def mix(self, *others: 'Channel') -> 'Channel':
        items = list(self._items)
        for o in others:
            items.extend(o._items)
        return Channel(items)

    def collect(self) -> list:
        return list(self._items)

    def view(self, label=''):
        for item in self._items:
            print(f"  {label}{item}")
        return self

# ── Simulate a two-step pipeline ─────────────────────────────────────────
reads_pairs = [
    ("sample1", ["data/sample1_R1.fastq.gz", "data/sample1_R2.fastq.gz"]),
    ("sample2", ["data/sample2_R1.fastq.gz", "data/sample2_R2.fastq.gz"]),
    ("sample3", ["data/sample3_R1.fastq.gz", "data/sample3_R2.fastq.gz"]),
]

reads_ch = Channel.fromFilePairs(reads_pairs)

# Simulate FASTP process output
def simulate_fastp(item):
    sample_id, reads = item
    trimmed = [r.replace('data/', 'trimmed/').replace('.fastq.gz', '.fq.gz')
               for r in reads]
    return (sample_id, trimmed)

trimmed_ch = reads_ch.map(simulate_fastp)

# Simulate BWA_MEM process output
def simulate_bwa(item):
    sample_id, reads = item
    bam = f"aligned/{sample_id}.bam"
    return (sample_id, bam)

bam_ch = trimmed_ch.map(simulate_bwa)

print("=== Nextflow channel simulation ===")
print("Input reads:")
reads_ch.view("  ")
print("\nAfter FASTP trimming:")
trimmed_ch.view("  ")
print("\nAfter BWA_MEM alignment:")
bam_ch.view("  ")
print(f"\nTotal BAMs: {len(bam_ch.collect())}")


---

## 6. nf-core Community Pipelines

### 6.1 What is nf-core?

**nf-core** (https://nf-co.re) is a community framework providing:

- **100+ curated pipelines** for common bioinformatics analyses
- Standardised structure, CI/CD testing, and versioned releases
- Reusable **nf-core/modules** (individual tool wrappers)
- The **nf-core tools** CLI for creating and linting pipelines

### 6.2 Key pipelines

| Pipeline | Use case |
|----------|----------|
| `nf-core/rnaseq` | Bulk RNA-seq: QC → alignment → quantification |
| `nf-core/sarek` | Germline + somatic variant calling (WGS/WES) |
| `nf-core/fetchngs` | Download FASTQ from SRA/ENA/DDBJ |
| `nf-core/atacseq` | ATAC-seq peak calling and differential accessibility |
| `nf-core/chipseq` | ChIP-seq peak calling |
| `nf-core/ampliseq` | 16S / ITS amplicon sequencing |
| `nf-core/scrnaseq` | Single-cell RNA-seq (multiple aligners) |
| `nf-core/mag` | Metagenome assembly and binning |
| `nf-core/methylseq` | Bisulfite methylation analysis |
| `nf-core/viralrecon` | Viral amplicon/metagenomics assembly |

### 6.3 Running an nf-core pipeline

```bash
# Install nf-core tools
pip install nf-core

# List available pipelines
nf-core list

# Download pipeline for offline use
nf-core download rnaseq --revision 3.14.0 --container singularity

# Run nf-core/rnaseq with Docker
nextflow run nf-core/rnaseq \
    -profile docker \
    --input samplesheet.csv \
    --genome GRCh38 \
    --outdir results/ \
    -r 3.14.0
```

### 6.4 Samplesheet format (CSV)

```csv
sample,fastq_1,fastq_2,strandedness
CTRL_1,data/ctrl_1_R1.fastq.gz,data/ctrl_1_R2.fastq.gz,auto
CTRL_2,data/ctrl_2_R1.fastq.gz,data/ctrl_2_R2.fastq.gz,auto
TREAT_1,data/treat_1_R1.fastq.gz,data/treat_1_R2.fastq.gz,auto
TREAT_2,data/treat_2_R1.fastq.gz,data/treat_2_R2.fastq.gz,auto
```

Validate your samplesheet before running:
```bash
nf-core launch rnaseq   # interactive parameter builder + validation
```

### 6.5 Custom configuration

#### Institutional config (`conf/institution.config`)
```groovy
// Tell nf-core where HPC resources are
process {
    executor = 'slurm'
    queue    = 'standard'
    memory   = { check_max( 8.GB * task.attempt, 'memory' ) }
    cpus     = { check_max( 4 * task.attempt, 'cpus' ) }
    time     = { check_max( 4.h * task.attempt, 'time' ) }
}
```

Shared institutional configs live in https://github.com/nf-core/configs — submit
a PR to add your HPC cluster.

#### Overriding specific processes
```groovy
// my_overrides.config
process {
    withName: 'STAR_ALIGN' {
        cpus   = 32
        memory = 60.GB
        time   = '6h'
    }
}
```

```bash
nextflow run nf-core/rnaseq -c my_overrides.config ...
```

### 6.6 Using nf-core modules in your own pipeline

```bash
# Create a new pipeline skeleton
nf-core create --name mypipeline --description "My analysis" --author "Your Name"

# Add a module from the nf-core/modules repository
nf-core modules install fastqc
nf-core modules install trimgalore

# List installed modules
nf-core modules list local

# Update a module
nf-core modules update fastqc
```

Installed modules appear in `modules/nf-core/fastqc/` with a
`main.nf` process and `meta.yml` documentation file.


In [ ]:
# ── nf-core samplesheet builder and validator ─────────────────────────────
import csv, io, re
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class RNAseqSample:
    sample: str
    fastq_1: str
    fastq_2: Optional[str]
    strandedness: str = 'auto'

    def validate(self):
        errors = []
        if not self.sample:
            errors.append("sample name is empty")
        if not re.match(r'^[A-Za-z0-9_\-]+$', self.sample):
            errors.append(f"sample name '{self.sample}' contains invalid characters")
        if not self.fastq_1.endswith(('.fastq.gz', '.fq.gz')):
            errors.append("fastq_1 must end with .fastq.gz or .fq.gz")
        if self.fastq_2 and not self.fastq_2.endswith(('.fastq.gz', '.fq.gz')):
            errors.append("fastq_2 must end with .fastq.gz or .fq.gz")
        if self.strandedness not in ('auto', 'forward', 'reverse', 'unstranded'):
            errors.append(f"strandedness must be auto/forward/reverse/unstranded")
        return errors

def build_rnaseq_samplesheet(samples: List[RNAseqSample]) -> str:
    """Build a valid nf-core/rnaseq CSV samplesheet string."""
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=['sample','fastq_1','fastq_2','strandedness'])
    writer.writeheader()
    for s in samples:
        writer.writerow({
            'sample':       s.sample,
            'fastq_1':      s.fastq_1,
            'fastq_2':      s.fastq_2 or '',
            'strandedness': s.strandedness,
        })
    return buf.getvalue()

# ── Demo: build and validate a samplesheet ───────────────────────────────
samples = [
    RNAseqSample("CTRL_1", "data/ctrl_1_R1.fastq.gz", "data/ctrl_1_R2.fastq.gz"),
    RNAseqSample("CTRL_2", "data/ctrl_2_R1.fastq.gz", "data/ctrl_2_R2.fastq.gz"),
    RNAseqSample("TREAT_1","data/treat_1_R1.fastq.gz","data/treat_1_R2.fastq.gz", strandedness="forward"),
    # Deliberately bad sample to show validation
    RNAseqSample("bad sample!", "data/bad.bam", None, strandedness="unknown"),
]

print("Validation results:")
for s in samples:
    errs = s.validate()
    status = '✓' if not errs else f'✗ {errs}'
    print(f"  {s.sample:<15}  {status}")

valid = [s for s in samples if not s.validate()]
print(f"\nValid samples: {len(valid)}/{len(samples)}")
print()
print("Generated samplesheet:")
print(build_rnaseq_samplesheet(valid))


---

## 7. Snakemake vs Nextflow Cheat-Sheet

| Feature | Snakemake | Nextflow (DSL2) |
|---------|-----------|-----------------|
| **Language** | Python + Makefile syntax | Groovy (JVM) |
| **Paradigm** | Rule-based (backward chaining) | Dataflow (channels) |
| **Config** | `config.yaml` + Python | `nextflow.config` (Groovy) |
| **Conda** | Native `conda:` directive | Via `conda` directive |
| **Containers** | `container:` (Singularity/Docker) | `container` directive |
| **Cluster** | Executor plugins | Executors (SLURM, SGE, AWS…) |
| **Community pipelines** | Snakemake Workflow Catalog | nf-core (100+ pipelines) |
| **Resume** | `--rerun-incomplete` | `-resume` (hash-based) |
| **Visualise DAG** | `--dag \| dot -Tpdf` | `dot` report via `-with-dag` |
| **Report** | `--report report.html` | `-with-report` |
| **Portability** | Good | Excellent (nf-core Docker images) |
| **Learning curve** | Lower (Python users) | Higher |
| **HPC adoption** | Very common | Very common |

### When to choose Snakemake
- Team is Python-native
- File-centric workflows (you think in terms of files, not streams)
- You want to prototype quickly
- Genome assembly, variant calling, bulk RNA-seq

### When to choose Nextflow
- Running an existing nf-core pipeline
- Need maximum portability across cloud providers
- Large-scale production pipelines with many samples
- Need fine-grained task-level resource control


---

## 8. Exercises

### Exercise 1 – Snakemake: RNA-seq pipeline

Write a Snakefile that:
1. Runs FastQC on raw FASTQ files
2. Trims adapters with fastp
3. Aligns with HISAT2
4. Quantifies with featureCounts
5. Runs MultiQC on all QC reports

Add a `config/config.yaml` with paths and parameters. Include `log:` and
`benchmark:` for every rule, and a `conda:` environment for each tool.


In [ ]:
# Scaffold — fill in the blanks
scaffold_rnaseq = '''
configfile: "config/config.yaml"

SAMPLES = config["samples"]
REF     = config["reference"]
GTF     = config["gtf"]

rule all:
    input:
        expand("results/counts/{sample}.txt", sample=SAMPLES),
        "results/qc/multiqc_report.html"

rule fastqc:
    input:  "data/{sample}_R1.fastq.gz"
    output: "results/qc/{sample}_fastqc.html"
    log:       "logs/fastqc/{sample}.log"
    benchmark: "benchmarks/fastqc/{sample}.tsv"
    conda:  "envs/qc.yaml"
    shell:  "fastqc {input} -o results/qc/ 2> {log}"

# Example: add trim rule (fastp)
# rule trim:
#     ...

# Example: add hisat2 rule
# rule hisat2:
#     ...

# Example: add featureCounts rule
# rule featurecounts:
#     ...

# Example: add multiqc rule
# rule multiqc:
#     ...
'''
print(scaffold_rnaseq)


### Exercise 2 – Snakemake: cluster profile

Create a SLURM profile directory at `profiles/slurm/config.yaml` that:
- Uses the `slurm` executor
- Sets default resources: 4 GB RAM, 2 CPUs, 2 hours
- Enables `--use-conda`
- Sets `--keep-going` and `--latency-wait 60`


In [ ]:
slurm_profile = '''
# profiles/slurm/config.yaml
executor: slurm
jobs: 100
use-conda: true
keep-going: true
latency-wait: 60
printshellcmds: true

default-resources:
  - mem_mb=4000
  - cpus_per_task=2
  - runtime=120          # minutes
  - slurm_account=mylab
  - slurm_partition=standard
'''
print("profiles/slurm/config.yaml content:")
print(slurm_profile)


### Exercise 3 – nf-core/sarek somatic variant calling

Using `nf-core/sarek`, write the `nextflow run` command to:
- Run somatic variant calling on a pair of tumour / normal WES samples
- Use Docker
- Apply the `hg38` genome
- Run Mutect2 and Strelka variant callers
- Save results to `results/somatic/`

Hints: check `nextflow run nf-core/sarek --help` and https://nf-co.re/sarek/parameters


In [ ]:
# Reference answer (fill in sample-specific paths)
sarek_command = '''
nextflow run nf-core/sarek \
    -profile docker \
    -r 3.4.4 \
    --input samplesheet_somatic.csv \
    --genome GATK.GRCh38 \
    --tools mutect2,strelka \
    --outdir results/somatic/ \
    --max_cpus 16 \
    --max_memory 64.GB
'''

# Samplesheet for somatic analysis
somatic_sheet = """patient,sample,lane,fastq_1,fastq_2,status
PT01,NORMAL,L001,data/normal_R1.fastq.gz,data/normal_R2.fastq.gz,0
PT01,TUMOR,L001,data/tumor_R1.fastq.gz,data/tumor_R2.fastq.gz,1
"""

print("Command:")
print(sarek_command)
print("Samplesheet (status 0=normal, 1=tumor):")
print(somatic_sheet)


---

## 9. Summary

| Topic | Key take-away |
|-------|--------------|
| Why workflow engines | DAG-based execution = automatic parallelism, resume, and reproducibility |
| Snakemake rules | `input/output/params/log/benchmark/threads/resources` directives |
| Snakemake conda | `conda:` per rule + `--use-conda` flag |
| Snakemake containers | `container:` directive + `--use-singularity` |
| Snakemake cluster | Executor plugins (slurm, googlebatch, aws-batch) + profiles |
| Nextflow DSL2 | Channels + Processes + Workflows; `-resume` for incremental re-runs |
| nf-core pipelines | 100+ production-ready pipelines; CSV samplesheets; `-profile docker` |
| nf-core modules | Reusable tool wrappers; `nf-core modules install <tool>` |
| Institutional configs | Shared HPC settings; submit to https://github.com/nf-core/configs |

### Next steps

- 📖 [Snakemake official tutorial](https://snakemake.readthedocs.io/en/stable/tutorial/tutorial.html)
- 📖 [Nextflow training](https://training.nextflow.io/)
- 📖 [nf-core usage guide](https://nf-co.re/usage/introduction)
- 🔬 [Snakemake Workflow Catalog](https://snakemake.github.io/snakemake-workflow-catalog/)
- 🔬 [nf-core pipelines list](https://nf-co.re/pipelines)


---

[← Previous: Single-Cell Analysis](./01_single_cell_scanpy.ipynb) | [Next: Testing and CI/CD →](./03_testing_cicd.ipynb)
